In [2]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [3]:
# standard
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [4]:
import cupy as cp
import pandas as pd
import numpy as np

In [5]:
# custom
from utils import *
import _run_constants as rc

# LOAD DATA

In [6]:
word_df, word_id_list, word_byte_list, word_byte_array, word_byte_to_word_dict = load_input_data()

In [8]:
word_byte_array_cp = cp.asarray(word_byte_array)

In [10]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

vibex beivx True
glyph ghlpy True
muntz mntuz True
dwarf adfrw True
jocks cjkos True


## EXAMPLES OF BYTE COMPARISONS

In [11]:
w1 = 'abhor'
w2 = 'cleft'
w3 = 'frown'
w1b = byte_encode_words(w1)
w2b = byte_encode_words(w2)
w3b = byte_encode_words(w3)

In [12]:
# no letters in common
w1b & w2b

0

In [13]:
# letters in common
w1b & w3b

147456

In [14]:
# bitwise or
w1b | w2b

673975

In [15]:
# this is the same as directly above
byte_encode_words('abhorcleft')

673975

In [16]:
byte_encode_words(ascii_lowercase)

67108863

# BUILD LEVEL 2

In [17]:
l2_list = np.full(shape = (10000000, 3), fill_value = -1, dtype = np.int32)
row_index = 0
found_values = set()
for w1_be, w2_be in combinations(word_byte_list, 2):
    if w1_be & w2_be == 0:   
        # they share no letters in common
        l2 = w1_be | w2_be                          
        
        l2_list[row_index, :] = np.array([w1_be, w2_be, l2], dtype = np.int32)
        found_values.add(l2)
        row_index += 1

# trim the data frame
l2_list = l2_list[:row_index, :]
print(l2_list.shape)
l2_df = pd.DataFrame(data = l2_list, columns = ['w1b', 'w2b', 'l2'])

(3213696, 3)


# BUILD LEVELS 3 THROUGH 5

In [20]:
# so, now, let's try computing all possible pairs
total_output = cp.full(shape = (1000000, 9), fill_value= -1, dtype = cp.int32)
row_index = 0
for i_row, row in l2_df.iterrows():    
    # levels 1 and 2
    w1b, w2b, l2 = row

    # indexer for l3
    positional_idx_l3_cp = (word_byte_array_cp & l2) == 0

    # l3 words with different letters
    output_array_w3b_cp = word_byte_array_cp[positional_idx_l3_cp]    

    # l3 accumulated letters
    output_array_l3_cp = output_array_w3b_cp | l2

    ## enumerate level 3
    for w3b, l3 in zip(output_array_w3b_cp, output_array_l3_cp):

        # build level 4

        # l4 idx
        positional_idx_l4_cp = (word_byte_array_cp & l3) == 0

        # words with different letters
        output_array_w4b_cp = word_byte_array_cp[positional_idx_l4_cp]    
        
        # accumulated letters
        output_array_l4_cp = output_array_w4b_cp | l3

        ## enumerate level 5
        for w4b, l4 in zip(output_array_w4b_cp, output_array_l4_cp):

            # build level 5

            # l5 idx
            positional_idx_l5_cp = (word_byte_array_cp & l4) == 0
            
            # words with different letters
            output_array_w5b_cp = word_byte_array_cp[positional_idx_l5_cp]    

            if output_array_w5b_cp.size > 0:
                    
                # accumulated letters
                output_array_l5_cp = output_array_w5b_cp | l4

                ## gather and combine the output
                for w5b, l5 in zip(output_array_w5b_cp, output_array_l5_cp):

                    temp_list = cp.array([w1b, w2b, w3b, w4b, w5b, l2, l3, l4, l5], dtype = cp.int32)
                    total_output[row_index, :] = temp_list                   
                    row_index += 1


    if i_row % 10000 == 0:
        # print the number of l2 iterations and the shape of the output
        print(i_row, row_index)
    


0 0
10000 0
20000 0
30000 0
40000 0
50000 0
60000 0
70000 0
80000 0
90000 0
100000 0
110000 0
120000 0
130000 0
140000 0
150000 0
160000 0
170000 0
180000 0
190000 0


TypeError: Implicit conversion to a NumPy array is not allowed. Please use `.get()` to construct a NumPy array explicitly.

# CREATE AND SAVE OUTPUT

In [ ]:
total_output = total_output[:row_index, :]
col_names = ['w1b', 'w2b', 'w3b', 'w4b', 'w5b', 'l2', 'l3', 'l4', 'l5']
l5_df = pd.DataFrame(data = total_output, columns = col_names)


In [ ]:
l5_df.shape

In [ ]:
l5_df.head()

In [ ]:
l5_df.tail()

In [ ]:
l5_df.to_csv(path_or_buf='l5.txt', sep = '\t', index = False)
